# Evaluation 01 — Real and optional synthetic test inference (four `timm` models)

Loads your actual saved `best_weights.pth` with `pretrained=False` and checks `run_config.json`; supports ResNet50, EfficientNet-B3, ConvNeXt Tiny and DINOv2 ViT-S/14 **as currently uploaded**. Never substitutes a random or ImageNet-only checkpoint for the trained model. Enable T4 GPU. **No retraining, resplitting or test-based tuning.**

In [ ]:
# Common Kaggle bootstrap: either attach DermSynReal source as Kaggle Input,
# or set CLONE_REPO_IF_MISSING=True and enable Kaggle Internet.
from pathlib import Path
import subprocess,sys,os,json
REPO_ROOT=Path("/kaggle/input/EDIT_REPO_DATASET/DermSynReal")
CLONE_REPO_IF_MISSING=True
EXPECTED_GIT_SHA=""  # Recommended: set a full pinned SHA for reproducible final runs.
if not (REPO_ROOT/"src/dermsynreal/evaluation.py").is_file():
    if not CLONE_REPO_IF_MISSING:
        raise FileNotFoundError("Attach the repository and change REPO_ROOT, or enable Git clone.")
    REPO_ROOT=Path("/kaggle/working/DermSynReal")
    if not (REPO_ROOT/".git").is_dir():
        subprocess.check_call(["git","clone","https://github.com/shanmukhdatta/DermSynReal.git",str(REPO_ROOT)])
if EXPECTED_GIT_SHA:
    if not (REPO_ROOT/".git").is_dir():
        raise ValueError("For pinned SHA verification attach a Git checkout or enable clone")
    actual=subprocess.check_output(["git","-C",str(REPO_ROOT),"rev-parse","HEAD"],text=True).strip()
    if actual!=EXPECTED_GIT_SHA:
        raise ValueError(f"Repository SHA {actual} differs from pinned SHA {EXPECTED_GIT_SHA}")
elif (REPO_ROOT/".git").is_dir():
    print("Current Git SHA:",subprocess.check_output(["git","-C",str(REPO_ROOT),"rev-parse","HEAD"],text=True).strip())
    print("WARNING: pin EXPECTED_GIT_SHA for final research outputs")
sys.path.insert(0,str(REPO_ROOT/"src"))
print("DermSynReal source:",REPO_ROOT)


## Edit mounted model and frozen test paths
The test manifest must contain `image_id,image_path,label,split,group_id` (`split=test`). Optional synthetic test additionally requires `prompt_family_id`, and the original **synthetic train-family manifest** so prompt-family leakage can be rejected.

In [ ]:
MODEL_ID="resnet50.a1_in1k"  # other choices in MODEL_ALLOWLIST
RUN_CONFIG="/kaggle/input/EDIT_MODEL_OUTPUT/run_config.json"
CHECKPOINT="/kaggle/input/EDIT_MODEL_OUTPUT/best_weights.pth"
REAL_TEST_MANIFEST="/kaggle/input/EDIT_REAL_SPLIT/real_test_manifest.csv"
REAL_IMAGE_ROOT="/kaggle/input/EDIT_REAL_IMAGE_ROOT"
SYNTH_TEST_MANIFEST=""  # optional path to prompt-family-held-out test CSV
SYNTH_IMAGE_ROOT=""
TRAIN_SYNTHETIC_MANIFEST=""  # required if SYNTH_TEST_MANIFEST supplied
OUTPUT_CSV="/kaggle/working/dermsynreal_eval_01/final_test_predictions.csv"
BATCH_SIZE=8
RUN_FINAL_TEST=False  # Unlock only after model, augmentation and checkpoint are finalized.

In [ ]:
from dermsynreal.inference import run_timm_inference, MODEL_ALLOWLIST
print("Supported exact existing models:",MODEL_ALLOWLIST)
if not RUN_FINAL_TEST:
    print("Test remains locked. Edit your Kaggle paths, freeze all training decisions, then set RUN_FINAL_TEST=True.")
else:
    import torch
    assert torch.cuda.is_available(),"Select Kaggle GPU accelerator for this evaluation notebook"
    result=run_timm_inference(model_id=MODEL_ID,run_config=RUN_CONFIG,weights_file=CHECKPOINT,
        real_manifest=REAL_TEST_MANIFEST,real_image_root=REAL_IMAGE_ROOT,
        synthetic_manifest=SYNTH_TEST_MANIFEST or None,synthetic_image_root=SYNTH_IMAGE_ROOT or None,
        train_synthetic_manifest=TRAIN_SYNTHETIC_MANIFEST or None,out_csv=OUTPUT_CSV,
        batch_size=BATCH_SIZE,num_workers=2,allow_test=True)
    print(json.dumps(result,indent=2))


In [ ]:
if RUN_FINAL_TEST:
    from dermsynreal.evaluation import read_predictions,source_metrics
    predictions=read_predictions(OUTPUT_CSV)
    results=source_metrics(predictions)
    for domain,summary in results.items():
        print(domain,"n =",summary["n"],"macro-F1 =",summary["macro_f1"],
              "accuracy =",summary["accuracy"],"missing true classes =",summary["missing_true_classes"])
    print("Saved CSV and SHA provenance under",Path(OUTPUT_CSV).parent)


**Important:** your older training notebook export CSV has no probability columns; this inference notebook adds all six `p_*` columns for calibration. Any synthetic-test run remains a separate, diagnostic endpoint.